In [37]:
from collections import deque
import copy

In [38]:
# Function to toggle the state of the lamp and its (+) neighbor lamps
def toggle_lights(matrix, x, y):
    # coppy the matrix
    new_matrix = copy.deepcopy(matrix)
    n = len(matrix)

    # Define the directions (current ,up, down, left, right)
    directions = [(0, 0), (1, 0), (-1, 0), (0, 1), (0, -1)]
    
    for dx, dy in directions:
        nx, ny = x + dx, y + dy
        # within the matrix
        if 0 <= nx < n and 0 <= ny < n:
            # Toggle the state (from on to off or vice versa)
            new_matrix[nx][ny] = 1 - new_matrix[nx][ny]
    
    return new_matrix

In [39]:
# check if all lamps are off
def all_off(matrix):
    return all(cell == 0 for row in matrix for cell in row)

## BFS

In [40]:
# BFS to find keys
def bfs(matrix):
    n = len(matrix)
    
    # Initial state
    start_state = (matrix, [])
    
    # Use a queue for BFS (deque is a double ended quque)
    queue = deque([start_state])
    
    # Use a set to store visited states
    # set is list with only unique value making it useful for 
    # checking if it already has been vidited
    visited = set()
    visited.add(str(matrix))
    
    # Count the number of visited nodes
    nodes_visited = 0
    
    while queue:

        # current_matrix will contain the initial matrix
        # path will contain the empty list
        current_matrix, path = queue.popleft()
        nodes_visited += 1

        # If all lamps are off, return the solution
        if all_off(current_matrix):
            return path, nodes_visited
        
        # Check all lamps for pressing their switches
        for i in range(n):
            for j in range(n):
                # New state after pressing the switch (i, j)
                new_matrix = toggle_lights(current_matrix, i, j)
                new_path = path + [(i, j)]
                
                # Check if the new state has been visited
                if str(new_matrix) not in visited:
                    visited.add(str(new_matrix))
                    queue.append((new_matrix, new_path))
    
    # If no solution is found
    return None, nodes_visited

# Sample input
initial_matrix0 = [
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1]
]

# Sample input
initial_matrix1 = [
    [0, 1, 1, 0],
    [1, 0, 0, 1],
    [1, 0, 0, 1],
    [0, 1, 1, 0]
]

# Sample input
initial_matrix2 = [
    [0, 1, 1, 0, 0],
    [1, 0, 0, 1, 0],
    [1, 0, 0, 1, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0]
]

input = initial_matrix0

# Run BFS algorithm
solution, nodes_visited = bfs(input)

if solution:
    print("Solution found with", len(solution), "steps:")
    print(solution)
else:
    print("No solution found.")

print("Nodes visited:", nodes_visited)

Solution found with 9 steps:
[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
Nodes visited: 512


## IDS

In [41]:
# Depth-Limited DFS function
def depth_limited_search(matrix, limit, path, visited):
    if all_off(matrix):  # Goal test
        return path, len(visited)
    
    if limit == 0:  # Depth limit reached
        return None, len(visited)
    
    n = len(matrix)
    
    # Visit all possible switches
    for i in range(n):
        for j in range(n):
            new_matrix = toggle_lights(matrix, i, j)  # Toggle the lights by pressing (i, j)
            new_path = path + [(i, j)]
            
            # Convert matrix to string to check visited state
            matrix_str = str(new_matrix)
            
            if matrix_str not in visited:
                visited.add(matrix_str)
                
                # Recursive call with reduced depth limit
                result, nodes_visited = depth_limited_search(new_matrix, limit - 1, new_path, visited)
                
                if result is not None:
                    return result, nodes_visited  # Solution found

    return None, len(visited)  # No solution at this depth

# IDS Function
def ids(matrix):
    depth = 0
    nodes_visited = 0
    
    while True:
        visited = set()  # To track visited states at each depth
        visited.add(str(matrix))  # Add initial state to visited
        
        # Perform depth-limited search
        result, new_nodes_visited = depth_limited_search(matrix, depth, [], visited)
        
        nodes_visited += new_nodes_visited  # Accumulate total nodes visited
        
        if result is not None:
            return result, nodes_visited  # Solution found
        
        depth += 1  # Increase depth limit for the next iteration


# Sample input
initial_matrix0 = [
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1]
]

# Sample input
initial_matrix1 = [
    [0, 1, 1, 0],
    [1, 0, 0, 1],
    [1, 0, 0, 1],
    [0, 1, 1, 0]
]

# Sample input
initial_matrix2 = [
    [0, 1, 1, 0, 0],
    [1, 0, 0, 1, 0],
    [1, 0, 0, 1, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0]
]

input = initial_matrix0

# Run the IDS algorithm on the sample matrix
result, nodes_visited = ids(input)

# Print the result in the required format
if result is None:
    print("No solution found.")
else:
    print(f"Solution found with {len(result)} steps:")
    print(result)
    print(f"Nodes visited: {nodes_visited}")

Solution found with 9 steps:
[(0, 0), (1, 2), (2, 1), (1, 1), (2, 2), (0, 1), (2, 0), (0, 2), (1, 0)]
Nodes visited: 1660


## unweighted and weighted A*

In [42]:
import heapq

# Heuristic: Count of On Lamps (consistent)
def heuristic_on_lamps(matrix):
    return sum(cell == 1 for row in matrix for cell in row)

# Heuristic: Manhattan Distance (more aggressive, might be inconsistent)
def heuristic_manhattan(matrix):
    n = len(matrix)
    total_distance = 0
    for i in range(n):
        for j in range(n):
            if matrix[i][j] == 1:  # Only count "on" lamps
                total_distance += min(i, n - i - 1) + min(j, n - j - 1)
    return total_distance

# Weighted A* algorithm
def weighted_a_star(matrix, heuristic, w):
    n = len(matrix)
    
    # Priority queue (min-heap), starting with the initial state
    start_state = (0, matrix, [])  # (f-value, matrix, path)
    queue = []
    heapq.heappush(queue, start_state)
    
    # Set to track visited states
    visited = set()
    visited.add(str(matrix))
    
    nodes_visited = 0  # Count of nodes visited

    while queue:
        f_value, current_matrix, path = heapq.heappop(queue)
        nodes_visited += 1

        # If all lamps are off, return the solution
        if all_off(current_matrix):
            return path, nodes_visited

        # Visit all possible switch presses
        for i in range(n):
            for j in range(n):
                # Generate new state after pressing the switch at (i, j)
                new_matrix = toggle_lights(current_matrix, i, j)
                new_path = path + [(i, j)]
                matrix_str = str(new_matrix)

                # If this state has not been visited
                if matrix_str not in visited:
                    visited.add(matrix_str)

                    # Calculate g(n) and h(n)
                    g_value = len(new_path)  # The path cost is the number of switches pressed
                    h_value = heuristic(new_matrix)  # The heuristic value
                    f_value = g_value + w * h_value  # f(n) = g(n) + w * h(n)

                    # Add the new state to the priority queue
                    heapq.heappush(queue, (f_value, new_matrix, new_path))

    # If no solution is found
    return None, nodes_visited

# Function to run Weighted A* with a specific heuristic and weight
def run_weighted_a_star(matrix, heuristic_name, w):
    if heuristic_name == "on_lamps":
        heuristic = heuristic_on_lamps
    elif heuristic_name == "manhattan":
        heuristic = heuristic_manhattan
    else:
        raise ValueError("Unknown heuristic")
    
    # Run the Weighted A* algorithm
    result, nodes_visited = weighted_a_star(matrix, heuristic, w)
    
    # Print result
    if result is None:
        print("No solution found.")
    else:
        print(f"Solution found with {len(result)} steps:")
        print(result)
        print(f"Nodes visited: {nodes_visited}")


# Sample input
initial_matrix0 = [
    [1, 0, 1],
    [1, 1, 0],
    [1, 0, 1]
]

# Sample input
initial_matrix1 = [
    [0, 1, 1, 0],
    [1, 0, 0, 1],
    [1, 0, 0, 1],
    [0, 1, 1, 0]
]

# Sample input
initial_matrix2 = [
    [0, 1, 1, 0, 0],
    [1, 0, 0, 1, 0],
    [1, 0, 0, 1, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0]
]

input = initial_matrix0

# Test the algorithm with both heuristics and different weights
print("\nRunning Weighted A* with 'on_lamps' heuristic and weight 1:")
run_weighted_a_star(input, "on_lamps", 1)

print("\nRunning Weighted A* with 'on_lamps' heuristic and weight 1.5:")
run_weighted_a_star(input, "on_lamps", 1.5)

print("\nRunning Weighted A* with 'on_lamps' heuristic and weight 2:")
run_weighted_a_star(input, "on_lamps", 2)

print("\nRunning Weighted A* with 'on_lamps' heuristic and weight 5:")
run_weighted_a_star(input, "on_lamps", 5)





Running Weighted A* with 'on_lamps' heuristic and weight 1:
Solution found with 5 steps:
[(0, 1), (0, 0), (1, 0), (2, 1), (2, 0)]
Nodes visited: 48

Running Weighted A* with 'on_lamps' heuristic and weight 1.5:
Solution found with 5 steps:
[(0, 1), (0, 0), (1, 0), (2, 1), (2, 0)]
Nodes visited: 44

Running Weighted A* with 'on_lamps' heuristic and weight 2:
Solution found with 5 steps:
[(0, 1), (0, 0), (1, 0), (2, 1), (2, 0)]
Nodes visited: 31

Running Weighted A* with 'on_lamps' heuristic and weight 5:
Solution found with 5 steps:
[(0, 1), (0, 0), (2, 1), (1, 0), (2, 0)]
Nodes visited: 62


In [43]:
print("\nRunning Weighted A* with 'manhattan' heuristic and weight 1:")
run_weighted_a_star(input, "manhattan", 1)

print("\nRunning Weighted A* with 'manhattan' heuristic and weight 1.5:")
run_weighted_a_star(input, "manhattan", 1.5)

print("\nRunning Weighted A* with 'manhattan' heuristic and weight 2:")
run_weighted_a_star(input, "manhattan", 2)

print("\nRunning Weighted A* with 'manhattan' heuristic and weight 5:")
run_weighted_a_star(input, "manhattan", 5)


Running Weighted A* with 'manhattan' heuristic and weight 1:
Solution found with 5 steps:
[(1, 0), (2, 1), (0, 1), (0, 0), (2, 0)]
Nodes visited: 77

Running Weighted A* with 'manhattan' heuristic and weight 1.5:
Solution found with 5 steps:
[(1, 0), (2, 1), (0, 1), (0, 0), (2, 0)]
Nodes visited: 86

Running Weighted A* with 'manhattan' heuristic and weight 2:
Solution found with 5 steps:
[(1, 0), (2, 1), (0, 1), (0, 0), (2, 0)]
Nodes visited: 88

Running Weighted A* with 'manhattan' heuristic and weight 5:
Solution found with 5 steps:
[(1, 0), (2, 1), (0, 1), (0, 0), (2, 0)]
Nodes visited: 167
